# Notebook 2 — Custom CNN: Training from Scratch

## Story
Before turning to pretrained models, we ask: *can we build a competitive classifier
entirely from scratch with our dataset?*

We explore two architectures of increasing depth and capacity:
1. **SimpleCNN** — 1 conv layer + 1 FC: the absolute minimum learnable model.
2. **SmallCNN** — 4 conv blocks (BN + ReLU + MaxPool) + a 3-layer MLP head.

Both are trained end-to-end on our training split.  
Spoiler: performance will be **limited by the amount of training data** — a problem
we address in later notebooks through transfer learning.

**Labels (12):** `pen, paper, book, clock, phone, laptop, chair, desk, bottle, keychain, backpack, calculator`

## 1. Imports & Setup

In [ ]:
import sys
sys.path.insert(0, "../")
sys.path.insert(0, "../experiments")

import torch
import torch.nn as nn
from pathlib import Path

from eval import LABEL_ORDER
from utils import (
    set_seed, load_dataset, split_dataset, subsample_subset,
    get_train_transform, get_eval_transform, build_dataloaders,
    plot_per_class_examples, plot_multilabel_examples,
    run_baselines, print_model_info,
    train_model, save_checkpoint, load_checkpoint, plot_training_history,
    collect_test_predictions, categorize_predictions,
    show_prediction_examples, plot_per_class_metrics,
    plot_confusion_matrices, show_saliency_examples,
    compute_multilabel_metrics, evaluate_predictor,
    NUM_LABELS,
)

SEED   = 42
set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 2. Config

In [ ]:
BASE_DATA_DIR   = "../data/aggregated"
IMAGE_SIZE      = 128
BATCH_SIZE      = 256
SPLIT           = [0.8, 0.1, 0.1]
USE_SUBSET      = False
SUBSET_FRACTION = 0.1
CHECKPOINT_DIR  = Path("../checkpoints")
print(f"Labels ({NUM_LABELS}): {LABEL_ORDER}")

## 3. Data Loading & Loaders

In [ ]:
full_dataset = load_dataset(BASE_DATA_DIR)
train_raw, val_raw, test_raw = split_dataset(full_dataset, SPLIT, SEED)

if USE_SUBSET:
    train_raw = subsample_subset(train_raw, SUBSET_FRACTION, SEED)
    val_raw   = subsample_subset(val_raw,   SUBSET_FRACTION, SEED + 1)
    test_raw  = subsample_subset(test_raw,  SUBSET_FRACTION, SEED + 2)

train_transform = get_train_transform(IMAGE_SIZE)
eval_transform  = get_eval_transform(IMAGE_SIZE)
train_loader, val_loader, test_loader = build_dataloaders(
    train_raw, val_raw, test_raw, train_transform, eval_transform, BATCH_SIZE
)
print(f"Train: {len(train_raw)}  Val: {len(val_raw)}  Test: {len(test_raw)}")

## 4. Model A — SimpleCNN (1 Conv + 1 FC)

The simplest possible CNN: one convolutional layer to extract local features,
then a linear classifier. This is our **absolute baseline** for learned models.

```
Input (3, 128, 128)
  → Conv(3→16, k=3) + ReLU + MaxPool(2)  → (16, 64, 64)
  → Flatten → Dropout(0.4) → Linear(16*64*64 → 12)
```

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_labels: int, image_size: int = IMAGE_SIZE):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),
        )
        flattened = 16 * (image_size // 2) * (image_size // 2)
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(flattened, num_labels),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.classifier(x)


def create_simple_cnn(num_labels: int) -> nn.Module:
    return SimpleCNN(num_labels=num_labels, image_size=IMAGE_SIZE)


print_model_info(create_simple_cnn(NUM_LABELS).to(DEVICE))

## 5. Train SimpleCNN

In [ ]:
SIMPLE_PATH = CHECKPOINT_DIR / "baseline_1cnn_1fc.pth"

best_state_simple, best_f1_simple, history_simple, epochs_simple = train_model(
    create_simple_cnn, NUM_LABELS, train_loader, val_loader, DEVICE,
    lr=1e-3, weight_decay=1e-3, max_epochs=30,
)
save_checkpoint(best_state_simple, SIMPLE_PATH)
print(f"\nSimpleCNN best val F1: {best_f1_simple:.4f}")
plot_training_history(history_simple, epochs_simple, "SimpleCNN", 1e-3, 1e-3)

## 6. Model B — SmallCNN (4 Conv Blocks + MLP Head)

A deeper architecture with Batch Normalization to stabilize training:

| Block | Output |
|---|---|
| Conv(3→16) + BN + ReLU + MaxPool | 64×64 |
| Conv(16→32) + BN + ReLU + MaxPool | 32×32 |
| Conv(32→64) + BN + ReLU + MaxPool | 16×16 |
| Flatten → FC(16384→256) → ReLU → Dropout → FC(256→128) → ... → FC→12 | — |

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_labels: int):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 128 → 64
            nn.Conv2d(3, 16, 3, padding="same"), nn.BatchNorm2d(16), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            # Block 2: 64 → 32
            nn.Conv2d(16, 32, 3, padding="same"), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            # Block 3: 32 → 16
            nn.Conv2d(32, 64, 3, padding="same"), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 256), nn.ReLU(inplace=True), nn.Dropout(0.2),
            nn.Linear(256, 128),          nn.ReLU(inplace=True), nn.Dropout(0.2),
            nn.Linear(128, 64),           nn.ReLU(inplace=True), nn.Dropout(0.2),
            nn.Linear(64, num_labels),
        )
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_in", nonlinearity="relu")
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode="fan_in", nonlinearity="relu")
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.classifier(self.features(x))


def create_small_cnn(num_labels: int) -> nn.Module:
    return SmallCNN(num_labels=num_labels)


print_model_info(create_small_cnn(NUM_LABELS).to(DEVICE))

## 7. Train SmallCNN

In [ ]:
SMALL_PATH = CHECKPOINT_DIR / "small_cnn_4conv_1fc.pth"

best_state_small, best_f1_small, history_small, epochs_small = train_model(
    create_small_cnn, NUM_LABELS, train_loader, val_loader, DEVICE,
    lr=1e-3, weight_decay=1e-3, max_epochs=50,
)
save_checkpoint(best_state_small, SMALL_PATH)
print(f"\nSmallCNN best val F1: {best_f1_small:.4f}")
plot_training_history(history_small, epochs_small, "SmallCNN", 1e-3, 1e-3)

## 8. Compare Both Models — Test Set

In [ ]:
import torch
from utils import compute_multilabel_metrics

models_to_eval = {
    "SimpleCNN": (create_simple_cnn, SIMPLE_PATH),
    "SmallCNN":  (create_small_cnn,  SMALL_PATH),
}

print(f"{'Model':<14} {'loss':>7} {'exact':>7} {'hamming':>8} {'IoU':>7} {'prec':>7} {'rec':>7} {'F1':>7}")
print("-" * 75)
for name, (fn, ckpt) in models_to_eval.items():
    m = load_checkpoint(fn, NUM_LABELS, ckpt, DEVICE)
    imgs, lbls, preds, probs = collect_test_predictions(m, test_loader, DEVICE)

    # re-collect logits for loss
    all_logits = []
    m.eval()
    with torch.no_grad():
        for images, _ in test_loader:
            all_logits.append(m(images.to(DEVICE)).cpu())
    logits = torch.cat(all_logits)
    met = compute_multilabel_metrics(lbls, preds, logits=logits)

    print(f"{name:<14} {met['loss']:>7.4f} {met['exact_match']:>7.4f} "
          f"{met['hamming_acc']:>8.4f} {met['mean_iou']:>7.4f} "
          f"{met['precision_micro']:>7.4f} {met['recall_micro']:>7.4f} "
          f"{met['f1_micro']:>7.4f}")

## 9. Prediction Analysis — Best Scratch Model

In [ ]:
# Use SmallCNN (deeper = better) for the analysis
model = load_checkpoint(create_small_cnn, NUM_LABELS, SMALL_PATH, DEVICE)
images, labels, preds, probs = collect_test_predictions(model, test_loader, DEVICE)
correct_idx, partial_idx, incorrect_idx = categorize_predictions(labels, preds)

show_prediction_examples(correct_idx,   images, labels, preds, "Fully Correct",     n=4)
show_prediction_examples(partial_idx,   images, labels, preds, "Partially Correct", n=4)
show_prediction_examples(incorrect_idx, images, labels, preds, "Fully Incorrect",   n=4)

In [ ]:
plot_per_class_metrics(labels, preds)
plot_confusion_matrices(labels, preds)

In [ ]:
show_saliency_examples(correct_idx,   images, labels, preds, model, "Fully Correct",     n=4, device=DEVICE)
show_saliency_examples(incorrect_idx, images, labels, preds, model, "Fully Incorrect",   n=4, device=DEVICE)

## 10. Conclusions

- Both models improve significantly over the non-learned baselines, but plateau
  quickly — their capacity outstrips the data available.
- **Data is the bottleneck**: CNNs trained from scratch need large datasets to
  learn robust visual representations. Ours is relatively small.
- Adding more layers (SmallCNN > SimpleCNN) helps, but with diminishing returns.

**Next:** We try deeper classical architectures — ResNet-style skip connections
and VGG-style stacks — both still trained from scratch (Notebook 3).